# Crawler ICD-10 (TT06) — icd.kcb.vn

Trang `https://icd.kcb.vn/icd-10-tt06/icd10-tt06` là một Angular SPA, dữ liệu không nằm trong HTML mà được gọi qua API riêng (đã reverse-engineer từ bundle JS `main.*.js`):

- **Base URL**: `https://ccs.whiteneuron.com/api/ICD10_TT06/`
- `GET root` → danh sách 22 chương bệnh (chapter)
- `GET childs/{model}?id={id}` → lấy các node con của 1 node hiện tại. `{model}` là loại của node **hiện tại** (`chapter` / `section` / `type` / ...), `id` là mã của node đó. Kết quả trả về node con kèm `model` của **cấp tiếp theo**.
- `GET data/{model}?id={id}` → lấy chi tiết đầy đủ của 1 node. Với node lá (thường là `disease`, `is_leaf=true`) có thêm các trường `include`, `exclude`, `note`.

Cây phân cấp quan sát được: `chapter` → `section` (mã 3 ký tự, vd A00) → `type` (mã 3 ký tự, vd A00) → `disease` (lá, mã 4 ký tự có dấu chấm, vd A00.0). Một số mã (vd nhóm ung thư C76-C97) được hiển thị lặp lại dưới nhiều node cha khác nhau — notebook khử trùng lặp theo `id` trước khi lấy chi tiết.

**Quan trọng — rate limit của server**: endpoint `data/` chỉ cho phép khoảng **~1 request/giây** (kiểu token-bucket). Vượt quá, server trả `HTTP 400` kèm body `{"error": "...gửi quá nhiều yêu cầu..."}` (không dùng mã 429 chuẩn). `api_get()` bên dưới tự nhận diện lỗi này và backoff-retry, đồng thời bước lấy chi tiết (Bước 2) chạy với concurrency thấp + có nghỉ giữa các request để tôn trọng giới hạn này.

Notebook này crawl toàn bộ cây, rồi lấy chi tiết include/exclude/note cho từng mã bệnh lá, và xuất ra CSV/Excel.

In [ ]:
import json
import os
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

BASE_URL = "https://ccs.whiteneuron.com/api/ICD10_TT06/"
OUTPUT_DIR = "data"
TREE_PATH = os.path.join(OUTPUT_DIR, "tree_nodes.json")
DETAIL_PATH = os.path.join(OUTPUT_DIR, "leaf_details.jsonl")
FINAL_CSV_PATH = os.path.join(OUTPUT_DIR, "icd10_tt06.csv")
FINAL_XLSX_PATH = os.path.join(OUTPUT_DIR, "icd10_tt06.xlsx")

# Chinh o day khi can:
MAX_WORKERS = 8          # concurrency cho buoc crawl cay (root/childs) - endpoint nay chiu tai tot hon
DETAIL_WORKERS = 2       # concurrency cho buoc lay chi tiet (data/) - endpoint nay bi gioi han ~1 req/s
DETAIL_PACE_SECONDS = 1.0  # nghi toi thieu giua cac request o buoc lay chi tiet
LIMIT_CHAPTERS = None    # vd: 2 -> chi crawl 2 chuong dau de test. None -> crawl toan bo.

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
class RateLimitedError(Exception):
    pass


def make_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=max(MAX_WORKERS, DETAIL_WORKERS) * 2)
    session.mount("https://", adapter)
    session.headers.update(
        {
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0 (compatible; icd10-tt06-crawler/1.0; +research)",
            "Referer": "https://icd.kcb.vn/",
        }
    )
    return session


SESSION = make_session()


def api_get(path: str, **params) -> dict:
    """GET voi backoff-retry rieng cho rate-limit cua app (tra ve HTTP 400 kem body {"error": ...}
    thay vi 429 chuan). Cac loi HTTP khac van raise binh thuong (va da duoc Retry adapter xu ly)."""
    last_resp = None
    for attempt in range(8):
        resp = SESSION.get(BASE_URL + path, params=params, timeout=20)
        last_resp = resp
        if resp.status_code == 400:
            try:
                body = resp.json()
            except ValueError:
                body = {}
            if isinstance(body, dict) and "error" in body:
                wait = min(1.5 * (attempt + 1), 15) + random.uniform(0, 0.5)
                time.sleep(wait)
                continue
        resp.raise_for_status()
        return resp.json()
    last_resp.raise_for_status()
    return last_resp.json()


def get_root() -> list[dict]:
    return api_get("root")["data"]


def get_childs(model: str, node_id: str) -> list[dict]:
    return api_get(f"childs/{model}", id=node_id)["data"]


def get_data(model: str, node_id: str) -> dict:
    return api_get(f"data/{model}", id=node_id)["data"]

## Bước 1 — Crawl cấu trúc cây (chapter → section → type → disease)

Duyệt theo từng tầng (BFS), mỗi tầng gọi song song `childs/{model}?id=...` cho tất cả node chưa phải lá. Kết quả được cache vào `data/tree_nodes.json`; nếu file đã tồn tại, ô dưới sẽ load lại thay vì crawl lại từ đầu (xoá file này nếu muốn crawl lại).

In [ ]:
def crawl_tree(max_workers: int = MAX_WORKERS, limit_chapters: int | None = LIMIT_CHAPTERS) -> list[dict]:
    if os.path.exists(TREE_PATH):
        with open(TREE_PATH, encoding="utf-8") as f:
            nodes = json.load(f)
        print(f"Da co cache: {len(nodes)} nodes tu {TREE_PATH} (xoa file de crawl lai)")
        return nodes

    root_nodes = get_root()
    if limit_chapters:
        root_nodes = root_nodes[:limit_chapters]
    for n in root_nodes:
        n["parent_path"] = []

    all_nodes: list[dict] = []
    failed: list[dict] = []
    frontier = root_nodes

    def fetch_childs_of(node: dict) -> list[dict]:
        time.sleep(random.uniform(0.02, 0.08))
        childs = get_childs(node["model"], node["id"])
        path = node["parent_path"] + [
            {"model": node["model"], "id": node["id"], "code": node["data"]["code"], "name": node["data"]["name"]}
        ]
        for c in childs:
            c["parent_path"] = path
        return childs

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        level = 0
        while frontier:
            all_nodes.extend(frontier)
            branches = [n for n in frontier if not n["is_leaf"]]
            print(f"level {level}: {len(frontier)} nodes ({len(branches)} can crawl tiep)")

            next_frontier: list[dict] = []
            futures = {pool.submit(fetch_childs_of, n): n for n in branches}
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"level {level}"):
                node = futures[fut]
                try:
                    next_frontier.extend(fut.result())
                except Exception as e:
                    failed.append({"model": node["model"], "id": node["id"], "error": str(e)})
            frontier = next_frontier
            level += 1

    if failed:
        print(f"CANH BAO: {len(failed)} node loi khi lay childs, xem bien `failed`")

    with open(TREE_PATH, "w", encoding="utf-8") as f:
        json.dump(all_nodes, f, ensure_ascii=False)
    print(f"Xong. Tong so node: {len(all_nodes)}")
    return all_nodes


nodes = crawl_tree()

In [ ]:
# Mot ma benh (id) co the xuat hien duoi nhieu node cha khac nhau (vd nhom C76-C97).
# Khu trung lap theo id, giu lan xuat hien dau tien, de moi ma chi bi crawl chi tiet dung 1 lan.
_seen_ids: set[str] = set()
leaf_nodes: list[dict] = []
for n in nodes:
    if n["is_leaf"] and n["id"] not in _seen_ids:
        _seen_ids.add(n["id"])
        leaf_nodes.append(n)

print(f"So node la (ma benh duy nhat): {len(leaf_nodes)} / tong {len(nodes)} node")

## Bước 2 — Lấy chi tiết từng mã bệnh (include / exclude / note)

Với mỗi node lá, gọi `data/{model}?id=...` để lấy thêm mô tả chi tiết. Do endpoint này bị giới hạn tốc độ chặt (~1 req/s), bước này chạy với concurrency thấp (`DETAIL_WORKERS`) và có nghỉ giữa các request; khi bị từ chối, `api_get` tự backoff-retry. Kết quả ghi nối tiếp vào `data/leaf_details.jsonl` — chỉ mã lấy **thành công** mới tính là xong, nên có thể dừng notebook giữa chừng và chạy lại ô này để tiếp tục (resumable), không mất tiến độ.

In [ ]:
def load_existing_details() -> dict[str, dict]:
    done: dict[str, dict] = {}
    if os.path.exists(DETAIL_PATH):
        with open(DETAIL_PATH, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                if "error" in rec:
                    continue  # lan truoc loi -> khong tinh la da xong, se retry
                done[rec["id"]] = rec
    return done


def crawl_details(leaves: list[dict], max_workers: int = DETAIL_WORKERS) -> dict[str, dict]:
    done = load_existing_details()
    todo = [n for n in leaves if n["id"] not in done]
    print(f"Da co {len(done)} ma benh (cache), can lay them {len(todo)} ma")

    if not todo:
        return done

    lock = threading.Lock()

    def fetch_one(node: dict) -> dict:
        time.sleep(DETAIL_PACE_SECONDS + random.uniform(0, 0.3))
        try:
            d = get_data(node["model"], node["id"])["data"]
            return {
                "id": node["id"],
                "code": d.get("code"),
                "name": d.get("name"),
                "include": d.get("include", ""),
                "exclude": d.get("exclude", ""),
                "note": d.get("note", ""),
            }
        except Exception as e:
            return {"id": node["id"], "error": str(e)}

    n_errors = 0
    with open(DETAIL_PATH, "a", encoding="utf-8") as out, ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(fetch_one, n) for n in todo]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="chi tiet ma benh"):
            rec = fut.result()
            with lock:
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out.flush()
            if "error" in rec:
                n_errors += 1
            else:
                done[rec["id"]] = rec

    if n_errors:
        print(f"CANH BAO: {n_errors} ma benh loi khi lay chi tiet lan nay. Chay lai o nay de retry.")
    return done


details = crawl_details(leaf_nodes)

## Bước 3 — Ghép dữ liệu và xuất CSV/Excel

In [ ]:
# Ten cac tang cha phia tren node la, dung ICD10_TT06 quan sat duoc: chapter -> section -> type -> disease.
# Neu do sau cay khac (truong hop dac biet), tu dong fallback ve ten level0/level1/...
ANCESTOR_LEVEL_NAMES = ["chapter", "section", "type"]


def flatten_leaves(leaves: list[dict]) -> list[dict]:
    records = []
    for n in leaves:
        rec = {}
        for i, anc in enumerate(n.get("parent_path", [])):
            label = ANCESTOR_LEVEL_NAMES[i] if i < len(ANCESTOR_LEVEL_NAMES) else f"level{i}"
            rec[f"{label}_code"] = anc["code"]
            rec[f"{label}_name"] = anc["name"]
        rec["disease_id"] = n["id"]
        rec["disease_code"] = n["data"]["code"]
        rec["disease_name"] = n["data"]["name"]
        records.append(rec)
    return records


base_df = pd.DataFrame(flatten_leaves(leaf_nodes))
detail_df = pd.DataFrame(
    [r for r in details.values() if "error" not in r]
)[["id", "include", "exclude", "note"]].rename(columns={"id": "disease_id"})

final_df = base_df.merge(detail_df, on="disease_id", how="left")
final_df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")
final_df.to_excel(FINAL_XLSX_PATH, index=False)

print(f"Da luu {len(final_df)} dong vao:\n- {FINAL_CSV_PATH}\n- {FINAL_XLSX_PATH}")
final_df.head(10)